<a href="https://colab.research.google.com/github/mayait/CursoAnalisisDatos_IA_2026/blob/main/sitio/labs/lab_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Laboratorio 15 · Ética y gobernanza del dato

La semana pasada terminaste con un modelo que decide a quién llama el equipo comercial. Funciona: le
gana a la línea base y genera dinero. Hoy no hay algoritmo nuevo. Hoy hay que **responder por lo que
ese modelo hace cuando decide sobre personas**, que es lo que va a pasar en cuanto salga del cuaderno.

Vas a auditar tu propio modelo, no uno de ejemplo. Vas a descubrir que trata a un grupo de clientes de
forma sistemáticamente distinta que a otro, que quitar la variable culpable **no arregla nada**, que la
variable menos importante del modelo es la que produce la disparidad más visible, y que la explicación
que le tienes que dar a un cliente concreto no se parece en nada al informe técnico. Al final escribes
la carta que recibiría esa persona. En papel. Con su nombre.

> **Hoy haces** · Reconstruyes el modelo de abandono del laboratorio 14 dentro de este cuaderno —sin
> depender de nada de fuera— y lo auditas (90 min). Mides desempeño por ciudad y por tipo de cliente,
> calculas la disparidad en tasas de falsos positivos y falsos negativos, y aplicas la regla de las
> cuatro quintas partes. Demuestras con un modelo auxiliar que el tipo de cliente se puede reconstruir
> desde el ticket medio, y compruebas qué pasa con la disparidad cuando eliminas la variable sensible.
> Calculas importancia global y explicación local, y **redactas la carta** que recibiría el cliente que
> el modelo dio por perdido. Cierras con la lista de verificación de privacidad y gobernanza del
> proyecto del grupo.
>
> **Entrega** · Borrador completo del cuaderno reproducible del proyecto, **con la sección de
> limitaciones y sesgos escrita**. Además: la tabla de desempeño por subgrupo del modelo del caso, la
> disparidad encontrada con su cifra, la demostración de la variable sustituta si existe, la carta a la
> persona afectada, y la lista de verificación de gobernanza llena.
> Nombre de archivo: `lab_15_apellido.ipynb`.

In [ ]:
# --- Setup del entorno ---
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Los datos de Comercial Andina viven en sitio/datos/
REPO = "https://github.com/mayait/CursoAnalisisDatos_IA_2026.git"
COPIA = Path("/content/CursoAnalisisDatos_IA_2026")
CANDIDATOS = [Path("../datos"), Path("datos"), Path("sitio/datos"),
              COPIA / "sitio" / "datos"]
DATOS = next((p for p in CANDIDATOS if p.exists()), None)
if DATOS is None:
    # En Colab el cuaderno llega solo: se trae el repositorio una sola vez.
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(COPIA)], check=True)
    DATOS = COPIA / "sitio" / "datos"

print("Setup completo ✓")
print(f"pandas {pd.__version__} · datos en {DATOS.resolve()}")

## 1. El modelo bajo auditoría, reconstruido aquí

Una auditoría que depende de que otro cuaderno esté abierto no es una auditoría. Este bloque repite,
comprimido, todo el laboratorio 14: la variable de abandono con su ventana de observación, las nueve
variables predictoras y la regresión logística. **Mismos datos, misma semilla, mismos números.**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score

ventas = pd.read_csv(DATOS / "ventas_limpias.csv", parse_dates=["fecha"])
clientes = pd.read_csv(DATOS / "clientes.csv", parse_dates=["fecha_alta"])
productos = pd.read_csv(DATOS / "productos.csv")
clientes["ciudad"] = clientes["ciudad"].str.strip().str.title().replace({"Guayaquíl": "Guayaquil"})

v = ventas.merge(productos[["producto_id", "costo_unitario"]], on="producto_id",
                 how="left", validate="m:1")
v["monto"] = v["cantidad"] * v["precio_unitario"] * (1 - v["descuento"])
v["margen"] = v["monto"] - v["cantidad"] * v["costo_unitario"]
compras = v[~v["es_devolucion"]]

HOY, VENTANA = v["fecha"].max(), 180
CORTE = HOY - pd.Timedelta(days=VENTANA)
dias_sin_comprar = (HOY - compras.groupby("cliente_id")["fecha"].max()).dt.days

historia = compras[compras["fecha"] <= CORTE]
g = historia.groupby("cliente_id")
X = pd.DataFrame({"frecuencia": g["factura_id"].nunique(), "monto": g["monto"].sum(),
                  "primera": g["fecha"].min(), "ultima": g["fecha"].max()})
X["margen"] = v[v["fecha"] <= CORTE].groupby("cliente_id")["margen"].sum().reindex(X.index)
X["recencia_corte"] = (CORTE - X["ultima"]).dt.days
X["antiguedad"] = (CORTE - X["primera"]).dt.days
X["ticket_medio"] = X["monto"] / X["frecuencia"]
X = X.join(clientes.set_index("cliente_id")[["razon_social", "ciudad", "tipo_cliente",
                                             "canal_captacion"]])
X["abandono"] = (dias_sin_comprar > VENTANA).reindex(X.index).astype(int)

NUMERICAS = ["recencia_corte", "frecuencia", "monto", "margen", "antiguedad", "ticket_medio"]
CATEGORICAS = ["ciudad", "tipo_cliente", "canal_captacion"]
UMBRAL = 0.50


def ajustar(columnas_categoricas, semilla=SEED):
    '''Ajusta la logística con el subconjunto de variables categóricas que se le pase.'''
    columnas = NUMERICAS + columnas_categoricas
    prep = ColumnTransformer([("num", StandardScaler(), NUMERICAS),
                              ("cat", OneHotEncoder(drop="first"), columnas_categoricas)])
    X_ent, X_pru, y_ent, y_pru = train_test_split(
        X[columnas], X["abandono"], test_size=0.30, stratify=X["abandono"], random_state=semilla)
    modelo = Pipeline([("prep", prep),
                       ("clf", LogisticRegression(max_iter=3000, random_state=semilla))])
    modelo.fit(X_ent, y_ent)
    prob = modelo.predict_proba(X_pru)[:, 1]
    auditoria = X.loc[X_pru.index].copy()
    auditoria["real"] = y_pru.values
    auditoria["prob"] = prob
    auditoria["pred"] = (prob >= UMBRAL).astype(int)
    return modelo, X_ent, X_pru, y_ent, y_pru, auditoria


modelo, X_ent, X_pru, y_ent, y_pru, aud = ajustar(CATEGORICAS)

print(f"clientes modelables {len(X):,} · abandono {X['abandono'].mean():.2%}")
print(f"conjunto de prueba  {len(aud):,} clientes\n")
print(f"área bajo la curva ROC : {roc_auc_score(aud['real'], aud['prob']):.4f}")
print(f"exactitud (umbral {UMBRAL:.2f})  : {accuracy_score(aud['real'], aud['pred']):.4f}")
print("\n← si estos dos números no coinciden con los del laboratorio 14, algo cambió y hay que verlo")

## 2. De dónde viene el sesgo

Un modelo no inventa desigualdades: **las hereda y las amplifica**. Antes de medir nada conviene tener
la lista de por dónde entra, porque cada entrada tiene un remedio distinto y ninguna se arregla
cambiando de algoritmo.

| fuente | qué pasa | dónde aparece en este modelo |
|---|---|---|
| **Datos históricos** | El pasado que aprende ya era desigual | Si el equipo comercial visitaba menos Loja, Loja compra menos y el modelo la marca como riesgo |
| **Variable objetivo mal elegida** | Se mide lo que se puede medir, no lo que importa | «180 días sin comprar» castiga a quien compra estacionalmente aunque nunca se haya ido |
| **Muestra no representativa** | Unos grupos tienen muchos más datos que otros | Quito aporta 632 clientes y Loja 147: el modelo se optimiza para Quito |
| **Variables sustitutas** | Se quita el atributo sensible y otra columna lo reconstruye | El ticket medio reconstruye el tipo de cliente casi perfectamente. Sección 4 |
| **Bucle de retroalimentación** | El modelo decide y su decisión genera los datos de mañana | A quien no se llama, se le pierde; el modelo «acierta» y se confirma a sí mismo |

Empezamos por la última columna: cuánta gente hay de cada grupo y cuánto abandona cada grupo **de
verdad**, antes de que el modelo opine.

In [ ]:
composicion = []
for var in ["ciudad", "tipo_cliente"]:
    t = X.groupby(var).agg(clientes=("abandono", "size"), tasa_real=("abandono", "mean"))
    t["% de la cartera"] = t["clientes"] / len(X) * 100
    t.index = [f"{var} · {i}" for i in t.index]
    composicion.append(t)
composicion = pd.concat(composicion)

print("Quién hay en los datos y cuánto abandona de verdad (sin modelo de por medio):\n")
print(composicion.to_string(float_format=lambda v: f"{v:,.4f}"))
print(f"\ntasa global de abandono: {X['abandono'].mean():.4f}")
print(f"razón entre el grupo que más abandona y el que menos: "
      f"{composicion['tasa_real'].max() / composicion['tasa_real'].min():.2f} veces")
print(f"el grupo más pequeño tiene {composicion['clientes'].min()} clientes "
      f"({composicion['% de la cartera'].min():.1f} % de la cartera): "
      "cualquier métrica suya es inestable")

📌 **El mayorista abandona al 15,51 % y el minorista al 37,55 %: 2,42 veces más.** Esa
diferencia es real, está en los datos y el modelo la va a aprender —es su trabajo—. La pregunta ética
no es si el modelo puede notar la diferencia, sino **qué hace con ella y a quién perjudica el error**.

Y una advertencia de método: Loja tiene 147 clientes, el 8,7 % de la cartera. En el conjunto de prueba
quedan unos cincuenta. Cualquier tasa calculada sobre cincuenta personas se mueve varios puntos con dos
casos. **Las métricas de subgrupos pequeños se reportan con el tamaño al lado, siempre.**

## 3. La auditoría: el mismo modelo, distintos resultados según a quién mires

Un modelo tiene una sola exactitud global y **tantas exactitudes como grupos haya**. La auditoría
consiste en romper la métrica por subgrupo y mirar las dos tasas de error por separado, porque no
duelen igual:

- **Tasa de falsos positivos** — de los clientes que se iban a quedar, ¿a qué proporción molestamos con
  una llamada de retención? Es el error que **incomoda**.
- **Tasa de falsos negativos** — de los clientes que se iban a ir, ¿a qué proporción dejamos ir sin
  intentar nada? Es el error que **abandona**, y en este modelo es el caro.

In [ ]:
def metricas_subgrupo(d):
    tn, fp, fn, tp = confusion_matrix(d["real"], d["pred"], labels=[0, 1]).ravel()
    return pd.Series({
        "n": len(d),
        "tasa real de abandono": d["real"].mean(),
        "tasa de selección": d["pred"].mean(),
        "exactitud": (tp + tn) / len(d),
        "TFP · falsos positivos": fp / (fp + tn) if (fp + tn) else np.nan,
        "TFN · falsos negativos": fn / (fn + tp) if (fn + tp) else np.nan,
        "precisión": tp / (tp + fp) if (tp + fp) else np.nan,
        "sensibilidad": tp / (tp + fn) if (tp + fn) else np.nan,
        "área ROC": roc_auc_score(d["real"], d["prob"]) if d["real"].nunique() > 1 else np.nan,
    })


global_ = metricas_subgrupo(aud).rename("TODA LA CARTERA")
por_tipo = aud.groupby("tipo_cliente").apply(metricas_subgrupo, include_groups=False)
por_ciudad = aud.groupby("ciudad").apply(metricas_subgrupo, include_groups=False)

print("Modelo global\n")
print(global_.to_string(float_format=lambda v: f"{v:,.4f}"))
print("\n\nPor tipo de cliente\n")
print(por_tipo.to_string(float_format=lambda v: f"{v:,.4f}"))
print("\n\nPor ciudad\n")
print(por_ciudad.to_string(float_format=lambda v: f"{v:,.4f}"))

⚠️ **La tasa de falsos negativos de los mayoristas es 1,0000.** De los 19 mayoristas del conjunto de
prueba que sí abandonaron, el modelo no señaló **ni uno solo**. Y su exactitud es 0,8077, la más alta
de toda la tabla: **el subgrupo donde el modelo parece funcionar mejor es aquel en el que no funciona
en absoluto**.

La explicación es aritmética y hay que entenderla porque va a pasar en tu proyecto. Como el mayorista
abandona poco (15,51 %), la probabilidad que el modelo le asigna casi nunca llega a 0,50, así que
predice «se queda» para todos y acierta el 82 % de las veces. Es la trampa de la exactitud del
laboratorio 14, ahora escondida dentro de un grupo.

La consecuencia de negocio no es abstracta: **los mayoristas son los clientes que más facturan de
Comercial Andina, y son exactamente los que el modelo nunca va a proteger.**

In [ ]:
def disparidad(tabla, columna):
    '''Razón entre el peor y el mejor grupo. 1 = trato idéntico.'''
    s = tabla[columna].dropna()
    return pd.Series({"mejor grupo": s.idxmin(), "valor mínimo": s.min(),
                      "peor grupo": s.idxmax(), "valor máximo": s.max(),
                      "razón de disparidad": s.max() / s.min() if s.min() > 0 else np.inf})


resumen = pd.DataFrame({
    (var, col): disparidad(tabla, col)
    for var, tabla in [("tipo_cliente", por_tipo), ("ciudad", por_ciudad)]
    for col in ["tasa de selección", "TFP · falsos positivos", "TFN · falsos negativos"]
}).T
print(resumen.to_string(float_format=lambda v: f"{v:,.4f}"))

# Regla de las cuatro quintas partes: ningún grupo debe recibir menos del 80 % de la tasa de
# selección del grupo más favorecido.
print("\n\nRegla de las cuatro quintas partes sobre la tasa de selección "
      "(a quién se le ofrece la campaña de retención):\n")
for var, tabla in [("tipo_cliente", por_tipo), ("ciudad", por_ciudad)]:
    s = tabla["tasa de selección"]
    razon = s / s.max()
    print(f"  {var}")
    for grupo, r in razon.sort_values().items():
        marca = "✗ por debajo de 0,80" if r < 0.80 else "✓"
        print(f"    {grupo:12s} tasa {s[grupo]:.4f} · razón contra el máximo {r:.4f}  {marca}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, tabla, titulo in [(axes[0], por_tipo, "por tipo de cliente"),
                          (axes[1], por_ciudad, "por ciudad")]:
    idx = np.arange(len(tabla))
    ax.barh(idx + 0.2, tabla["TFP · falsos positivos"], 0.4, color="#4C72B0",
            label="falsos positivos · molesta")
    ax.barh(idx - 0.2, tabla["TFN · falsos negativos"], 0.4, color="#C44E52",
            label="falsos negativos · abandona")
    ax.set_yticks(idx, [f"{i}\n(n={int(tabla.loc[i, 'n'])})" for i in tabla.index], fontsize=9)
    ax.set_xlim(0, 1.05)
    ax.set_xlabel("tasa de error")
    ax.set_title(titulo, fontsize=11)
    ax.legend(fontsize=8, loc="lower right")
fig.suptitle("El mismo modelo deja escapar al 100 % de los mayoristas y al 59 % de los minoristas",
             fontsize=12, y=1.03)
plt.tight_layout()
plt.show()

📌 **Las tres disparidades, con su cifra:**

| medida | grupo mejor tratado | grupo peor tratado | razón |
|---|---|---|---|
| Tasa de selección · tipo de cliente | Minorista 0,2333 | Mayorista 0,0096 | **24,3 veces** |
| Falsos negativos · tipo de cliente | Minorista 0,5918 | Mayorista 1,0000 | 1,69 veces |
| Falsos positivos · ciudad | Cuenca 0,0408 | Loja 0,1389 | **3,40 veces** |
| Falsos negativos · ciudad | Loja 0,3571 | Manta 0,7368 | 2,06 veces |

La **regla de las cuatro quintas partes** —que viene del derecho laboral estadounidense y se usa como
primer filtro de disparidad— dice que ningún grupo debería recibir el beneficio a menos del 80 % de la
tasa del grupo más favorecido. Aquí el mayorista recibe la campaña de retención al **4,1 %** de la tasa
del minorista, y Cuenca al 44,0 % de la tasa de Loja. **Las dos fallan la regla, y una falla por veinte
veces.**

Antes de indignarse conviene decidir cuál de las dos disparidades importa, porque **no se pueden
igualar todas a la vez** —hay una imposibilidad matemática demostrada—: si dos grupos tienen tasas
reales distintas, igualar la tasa de selección desiguala la precisión y viceversa. Elegir cuál se
iguala es una **decisión de negocio y de política, no estadística**, y hay que escribirla.

En este caso la decisión es fácil de argumentar: la campaña de retención es un **beneficio** (alguien
te llama para cuidarte), y dejar fuera al 100 % de los mayoristas que se van es un error caro y un
trato injusto. Se ataca la tasa de falsos negativos.

## 4. Quitar la variable sensible no elimina el sesgo

La reacción natural es: *«si el problema es el tipo de cliente, lo quito del modelo»*. Suena razonable
y casi nunca funciona, porque la información sigue en los datos repartida entre otras columnas. Se
llama **variable sustituta**, y la forma de demostrar que existe es tratar de reconstruir la variable
sensible desde las demás.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(5, shuffle=True, random_state=SEED)


def se_puede_reconstruir(objetivo, predictoras_cat):
    '''¿Se puede predecir la variable sensible desde el resto? Sí = hay sustituta.'''
    y = X[objetivo]
    Xp = X[NUMERICAS + predictoras_cat]
    prep = ColumnTransformer([("num", StandardScaler(), NUMERICAS),
                              ("cat", OneHotEncoder(drop="first"), predictoras_cat)])
    modelo_aux = Pipeline([("prep", prep),
                           ("clf", LogisticRegression(max_iter=3000, random_state=SEED))])
    base = cross_val_score(DummyClassifier(strategy="most_frequent"), Xp, y,
                           cv=skf, scoring="accuracy").mean()
    acc = cross_val_score(modelo_aux, Xp, y, cv=skf, scoring="accuracy").mean()
    return pd.Series({"línea base (predecir el grupo mayoritario)": base,
                      "modelo auxiliar": acc, "mejora": acc - base})


prueba = pd.DataFrame({
    "tipo_cliente desde el resto": se_puede_reconstruir("tipo_cliente",
                                                        ["ciudad", "canal_captacion"]),
    "ciudad desde el resto": se_puede_reconstruir("ciudad",
                                                  ["tipo_cliente", "canal_captacion"]),
}).T
print(prueba.to_string(float_format=lambda v: f"{v:.4f}"))

solo_ticket = Pipeline([("s", StandardScaler()),
                        ("c", LogisticRegression(max_iter=3000, random_state=SEED))])
acc_ticket = cross_val_score(solo_ticket, X[["ticket_medio"]],
                             X["tipo_cliente"], cv=skf, scoring="accuracy").mean()
print(f"\nY con UNA sola columna, el ticket medio: {acc_ticket:.4f} de exactitud")
print(f"  ticket medio mediano · mayorista {X.loc[X.tipo_cliente == 'Mayorista', 'ticket_medio'].median():,.2f}"
      f"  ·  minorista {X.loc[X.tipo_cliente == 'Minorista', 'ticket_medio'].median():,.2f}")

📌 **El tipo de cliente se reconstruye con un 99,88 % de exactitud desde las demás variables, y con un
99,82 % usando solo el ticket medio.** No hace falta ninguna técnica sofisticada: el mayorista tiene un
ticket mediano de 477,22 dólares y el minorista de 21,58. La columna `tipo_cliente` es, literalmente,
redundante.

Y el contraejemplo honesto, que es igual de importante: **la ciudad NO se puede reconstruir.** El
modelo auxiliar saca 0,3693 de exactitud contra 0,3746 de la línea base: **peor que adivinar «Quito»
siempre**. En estos datos la ciudad no tiene sustituta.

Conclusión operativa: **hay que probarlo, no suponerlo.** Una variable sensible tiene sustituta o no la
tiene, y la diferencia decide qué remedio aplicar. Ahora la prueba definitiva: quitar la columna del
modelo y volver a auditar.

In [ ]:
_, _, _, _, _, aud_sin_tipo = ajustar(["ciudad", "canal_captacion"])
_, _, _, _, _, aud_sin_nada = ajustar(["canal_captacion"])

comparativa = pd.DataFrame({
    "modelo completo": por_tipo["TFN · falsos negativos"],
    "sin la columna tipo_cliente": (aud_sin_tipo.groupby("tipo_cliente")
                                    .apply(metricas_subgrupo, include_groups=False)
                                    ["TFN · falsos negativos"]),
    "sin tipo_cliente NI ciudad": (aud_sin_nada.groupby("tipo_cliente")
                                   .apply(metricas_subgrupo, include_groups=False)
                                   ["TFN · falsos negativos"]),
})
print("Tasa de falsos negativos por tipo de cliente, con y sin la variable sensible\n")
print(comparativa.to_string(float_format=lambda v: f"{v:.4f}"))
print("\nÁrea bajo la curva ROC de cada modelo (el desempeño global casi no se mueve):")
for nombre, d in [("modelo completo", aud), ("sin tipo_cliente", aud_sin_tipo),
                  ("sin tipo_cliente ni ciudad", aud_sin_nada)]:
    print(f"  {nombre:28s} {roc_auc_score(d['real'], d['prob']):.4f}  ·  "
          f"disparidad TFN "
          f"{(d.groupby('tipo_cliente').apply(metricas_subgrupo, include_groups=False)['TFN · falsos negativos']).max() / (d.groupby('tipo_cliente').apply(metricas_subgrupo, include_groups=False)['TFN · falsos negativos']).min():.4f}")

## 5. Explicabilidad: explicar el modelo no es explicar la decisión

Son dos cosas distintas y las dos hacen falta:

- **Explicación global** — qué variables mueven el modelo en conjunto. Sirve para el comité de riesgos
  y para decidir si el modelo es defendible.
- **Explicación local** — por qué *este* cliente recibió *esta* decisión. Sirve para la persona
  afectada, y es la única que exige la ley en casi todas partes.

Un modelo puede tener una explicación global impecable y una explicación local indefendible. Empezamos
por la global, medida como debe medirse: **cuánto empeora el modelo si desordeno esa columna**.

In [ ]:
from sklearn.inspection import permutation_importance

imp = permutation_importance(modelo, X_pru, y_pru, n_repeats=20,
                             random_state=SEED, scoring="roc_auc")
importancia = pd.DataFrame({
    "variable": X_pru.columns,
    "caída del área ROC al desordenarla": imp.importances_mean,
    "desviación": imp.importances_std,
}).sort_values("caída del área ROC al desordenarla", ascending=False)

print(importancia.to_string(index=False, float_format=lambda v: f"{v:+.5f}"))

fig, ax = plt.subplots(figsize=(9, 3.6))
colores = ["#C44E52" if x <= 0 else "#4C72B0"
           for x in importancia["caída del área ROC al desordenarla"]]
ax.barh(importancia["variable"], importancia["caída del área ROC al desordenarla"], color=colores)
ax.invert_yaxis()
ax.axvline(0, color="grey", linewidth=1)
ax.set_xlabel("caída del área bajo la curva ROC")
ax.set_title("Dos variables sostienen el modelo; la ciudad no aporta nada y aun así discrimina",
             fontsize=11)
plt.tight_layout()
plt.show()

📌 **La ciudad tiene importancia −0,00151: desordenarla mejora el modelo.** Es ruido. Y sin embargo,
en la sección 3, esa misma variable produjo una disparidad de 3,40 veces en la tasa de falsos positivos
entre Cuenca y Loja.

Esa frase merece leerse dos veces, porque desmonta el argumento más común en defensa de un modelo:
*«pero si esa variable casi no pesa»*. **El peso de una variable en el modelo y el daño que produce en
un grupo son dos cosas distintas.** Una variable irrelevante puede ser la que decide quién recibe la
llamada en una ciudad pequeña, porque en ese grupo hay pocos casos y cualquier empujón cambia el
resultado.

La consecuencia práctica es inmediata y va al informe: **la ciudad se saca del modelo.** No perdemos
desempeño —el modelo sin ciudad ni tipo de cliente sube de 0,7589 a 0,7620 de área ROC— y ganamos una
fuente de disparidad menos. Cuando quitar una variable no cuesta nada, se quita.

Ahora la explicación local. En una regresión logística la contribución de cada variable a un cliente
concreto es su valor estandarizado por su coeficiente, y todas suman los log-momios de la decisión.

In [ ]:
# El cliente que vamos a explicar: el FALSO POSITIVO con la probabilidad más alta.
# Es decir, la persona a la que el modelo dio por perdida y que en realidad volvió a comprar.
falsos_positivos = aud[(aud["real"] == 0) & (aud["pred"] == 1)].sort_values("prob", ascending=False)
CLIENTE = falsos_positivos.index[0]
ficha = X.loc[[CLIENTE]]

prep_ajustado = modelo.named_steps["prep"]
z = np.asarray(prep_ajustado.transform(ficha[NUMERICAS + CATEGORICAS]))[0]
nombres = [n.split("__")[1] for n in prep_ajustado.get_feature_names_out()]
coef = modelo.named_steps["clf"].coef_[0]
intercepto = modelo.named_steps["clf"].intercept_[0]

local = pd.DataFrame({"variable": nombres, "valor estandarizado": z, "coeficiente": coef,
                      "contribución": z * coef})
local = local[local["contribución"].abs() > 1e-9].sort_values("contribución", ascending=False)

print(f"falsos positivos en el conjunto de prueba: {len(falsos_positivos)} de {len(aud)} clientes\n")
print(f"Cliente auditado: {CLIENTE} · {ficha['razon_social'].iloc[0]}")
print(f"  ciudad {ficha['ciudad'].iloc[0]} · {ficha['tipo_cliente'].iloc[0]} · "
      f"captado por {ficha['canal_captacion'].iloc[0]}")
print(f"  {int(ficha['recencia_corte'].iloc[0])} días sin comprar al corte · "
      f"{int(ficha['frecuencia'].iloc[0])} factura(s) · "
      f"{ficha['monto'].iloc[0]:,.2f} de compra acumulada")
print(f"\nprobabilidad de abandono que le asignó el modelo: {aud.loc[CLIENTE, 'prob']:.4f}")
print(f"¿abandonó de verdad?: {'sí' if aud.loc[CLIENTE, 'real'] else 'NO — el modelo se equivocó'}\n")
print(local.to_string(index=False, float_format=lambda v: f"{v:+.4f}"))
print(f"\nsuma de contribuciones {local['contribución'].sum():+.4f} "
      f"+ intercepto {intercepto:+.4f} = {local['contribución'].sum() + intercepto:+.4f} log-momios")
print(f"→ probabilidad = 1/(1+e^-x) = "
      f"{1 / (1 + np.exp(-(local['contribución'].sum() + intercepto))):.4f}")

fig, ax = plt.subplots(figsize=(9, 3.4))
top = local.head(6)
ax.barh(top["variable"], top["contribución"],
        color=["#C44E52" if c > 0 else "#4C72B0" for c in top["contribución"]])
ax.invert_yaxis()
ax.axvline(0, color="grey", linewidth=1)
ax.set_xlabel("contribución a los log-momios de abandono")
ax.set_title(f"Por qué el modelo dio por perdido a {CLIENTE}: casi todo son días sin comprar",
             fontsize=11)
plt.tight_layout()
plt.show()

### La carta

Un cliente tiene derecho a saber por qué lo trataron distinto, y ese derecho no se satisface con una
tabla de coeficientes. La regla de redacción es sencilla y dura: **si la explicación no le dice a la
persona qué puede hacer para cambiar el resultado, no es una explicación, es una notificación.**

La celda siguiente arma la carta con los números que acabamos de calcular. Ni una cifra está escrita a
mano.

In [ ]:
dias = int(ficha["recencia_corte"].iloc[0])
facturas = int(ficha["frecuencia"].iloc[0])
prob = aud.loc[CLIENTE, "prob"]
mediana_dias = int(X["recencia_corte"].median())
mediana_facturas = int(X["frecuencia"].median())
principales = local.head(3)["variable"].tolist()

carta = f'''
Guayaquil, {HOY:%d} de julio de {HOY:%Y}

Estimado cliente {CLIENTE} — {ficha['razon_social'].iloc[0]}

Le escribimos porque Comercial Andina usa un sistema automático para decidir a qué clientes
contacta su equipo comercial con ofertas de reactivación. Su cuenta quedó clasificada dentro
del grupo de "riesgo alto de inactividad", con una probabilidad estimada de {prob:.0%}.

Estos son los tres factores que más pesaron en esa clasificación, en orden:

  1. Días transcurridos desde su última compra: {dias}. La mediana de nuestros clientes
     es de {mediana_dias} días. Este factor aporta {local.iloc[0]['contribución']:+.2f} de los
     {local['contribución'].sum():+.2f} puntos totales de la evaluación.
  2. Número de compras registradas en el periodo analizado: {facturas}, frente a una mediana
     de {mediana_facturas}.
  3. Su segmento comercial (minorista), que en nuestra base tiene una tasa de inactividad
     más alta que el segmento mayorista.

Qué significa esto en la práctica: su cuenta entra en la lista de contacto prioritario. NO
implica ninguna restricción de crédito, de precio ni de condiciones comerciales.

Qué puede hacer si no está de acuerdo:
  · Cualquier compra registrada reinicia el primer factor, que es el que más pesa.
  · Puede pedir la revisión manual de su caso por una persona escribiendo a
    datos@comercialandina.ec. Un analista revisa la clasificación y responde en 15 días
    hábiles.
  · Puede pedir copia de los datos que usamos y solicitar la corrección de los que estén mal.

Responsable del sistema: Gerencia Comercial. Última revisión del modelo: {HOY:%d-%m-%Y}.
'''
print(carta)

⚠️ **Y ahora el detalle que convierte este ejercicio en un problema real: este cliente no abandonó.**
El modelo le asignó una probabilidad del 86 % y volvió a comprar. Es un falso positivo, uno de los 35
que hay en el conjunto de prueba.

En este caso el daño es pequeño —recibe una llamada que no necesitaba— y por eso la carta se puede
escribir con tranquilidad. **Cambia la decisión y cambia todo.** Con exactamente el mismo modelo, la
misma probabilidad y la misma carta:

- Si la decisión fuera *«a este cliente no le damos crédito a 30 días»*, el 14 % de probabilidad de
  equivocarse deja de ser una molestia y se convierte en un negocio que no puede comprar.
- Si fuera *«a este cliente le subimos el precio porque se va a ir igual»*, el modelo estaría
  **provocando** el abandono que predijo. El bucle de retroalimentación de la sección 2, en directo.

Las tres preguntas que hay que contestar por escrito antes de desplegar cualquier modelo que decide
sobre personas:

1. **¿Qué pasa cuando se equivoca?** No en promedio: en el caso concreto de esta persona.
2. **¿La persona puede saberlo y puede apelar?** Si la respuesta es no, el modelo no está listo.
3. **¿Puede hacer algo para cambiar el resultado?** Si la variable que más pesa es una que no puede
   modificar, la explicación es una burla.

## 6. Privacidad y gobernanza: la lista de verificación

La ética de datos no se resuelve con buenas intenciones sino con una lista aburrida que alguien firma.
Esta es la del curso: **cada grupo la llena para su propio proyecto**, con evidencia, no con «sí».

In [ ]:
VERIFICACION = [
    ("Privacidad", "¿Qué dato personal necesitamos de verdad para que el modelo funcione?",
     "Lista de variables usadas y una línea por variable justificando por qué está"),
    ("Privacidad", "¿Hay algún dato que recogemos y no usamos?",
     "Columnas del archivo original que NO entran al modelo, y decisión de borrarlas o no"),
    ("Privacidad", "¿Se puede identificar a una persona con el resultado?",
     "Prueba de reidentificación: ¿basta ciudad + segmento + monto para señalar a alguien?"),
    ("Privacidad", "¿Cuánto tiempo se guardan los datos y quién lo decidió?",
     "Plazo de retención en meses y nombre del área que lo aprobó"),
    ("Privacidad", "¿Quién puede ver la tabla con nombres y quién solo los agregados?",
     "Lista de roles con acceso y dónde vive el archivo"),
    ("Sesgo", "¿Medimos el desempeño por subgrupo?",
     "Tabla de métricas por subgrupo con el tamaño de cada grupo"),
    ("Sesgo", "¿Cuál es la disparidad máxima encontrada y en qué métrica?",
     "Razón de disparidad con su cifra y la regla de las cuatro quintas partes aplicada"),
    ("Sesgo", "¿Probamos si las variables sensibles tienen sustitutas?",
     "Exactitud del modelo auxiliar contra su línea base"),
    ("Sesgo", "¿Qué grupo sale perjudicado y cuánto cuesta el error para él?",
     "Nombre del grupo, tasa de error y traducción a dinero o a acceso"),
    ("Explicabilidad", "¿Podemos explicar una decisión concreta en lenguaje no técnico?",
     "La carta redactada para un caso real del conjunto de prueba"),
    ("Explicabilidad", "¿La persona puede hacer algo para cambiar el resultado?",
     "Lista de variables accionables por la persona frente a las que no lo son"),
    ("Gobernanza", "¿Quién responde cuando el modelo se equivoca?",
     "Nombre del cargo, no de la persona, y su firma"),
    ("Gobernanza", "¿Cada cuánto se revisa y con qué señal se apaga?",
     "Frecuencia de revisión y el umbral de degradación que dispara la alarma"),
    ("Gobernanza", "¿Existe un canal de apelación humana?",
     "Correo o proceso concreto, y el plazo de respuesta comprometido"),
    ("Gobernanza", "¿Está registrado qué versión del modelo tomó cada decisión?",
     "Bitácora con fecha, versión y umbral usado"),
]

checklist = pd.DataFrame(VERIFICACION, columns=["área", "pregunta", "evidencia que hay que adjuntar"])
checklist["estado"] = "PENDIENTE"
checklist["responsable"] = ""
checklist["nota"] = ""

# Ejemplo: las que este cuaderno ya deja resueltas para el modelo de Comercial Andina.
resueltas = [5, 6, 7, 8, 9]
checklist.loc[resueltas, "estado"] = "LISTO"
checklist.loc[resueltas, "responsable"] = "equipo de análisis"
checklist.loc[5, "nota"] = f"sección 3 · {len(por_ciudad)} ciudades y 2 tipos de cliente"
checklist.loc[6, "nota"] = (f"tasa de selección mayorista/minorista = "
                            f"{por_tipo.loc['Mayorista', 'tasa de selección'] / por_tipo.loc['Minorista', 'tasa de selección']:.4f}")
checklist.loc[7, "nota"] = f"tipo_cliente reconstruible con {acc_ticket:.4f} solo desde el ticket medio"
checklist.loc[8, "nota"] = "mayoristas: TFN 1,0000 · son los que más facturan"
checklist.loc[9, "nota"] = f"carta de la sección 5 para el cliente {CLIENTE}"

print(f"{len(checklist)} puntos · {(checklist['estado'] == 'LISTO').sum()} resueltos en este cuaderno "
      f"· {(checklist['estado'] == 'PENDIENTE').sum()} pendientes para el grupo\n")
print(checklist.to_string(index=False, max_colwidth=58))
print("\n\nPuntos pendientes por área:")
print(checklist[checklist["estado"] == "PENDIENTE"]["área"].value_counts().to_string())

### 🌶️ Ejercicio 1 — Guiado

Audita el modelo por **canal de captación**, que es la variable que no miramos hoy. Entrega: (a) la
tabla completa de `metricas_subgrupo` por canal con el tamaño de cada grupo, (b) la razón de disparidad
en falsos positivos y en falsos negativos, (c) la regla de las cuatro quintas partes aplicada a la tasa
de selección, y (d) una frase que diga si hay que hacer algo o no, con el número que la sostiene.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: aud.groupby("canal_captacion").apply(metricas_subgrupo, include_groups=False)
# Pista 2: la función disparidad() de la sección 3 ya hace el cálculo de la razón
# Pista 3: un grupo con menos de 100 clientes en prueba da métricas inestables. Antes de denunciar
#          una disparidad, comprueba cuántos casos la sostienen: si son cuatro, di que son cuatro

### 🔥 Desafío

**Arregla la disparidad y paga el precio.** El modelo deja escapar al 100 % de los mayoristas que se
van. Prueba tres remedios y compáralos en una sola tabla:

1. **Umbral por grupo**: un corte distinto para mayoristas —el que iguale su tasa de selección a la de
   los minoristas— y el 0,50 para el resto.
2. **Pesos de clase**: `LogisticRegression(class_weight="balanced")`.
3. **Un modelo por segmento**: uno para mayoristas y otro para minoristas.

Para cada uno reporta: área bajo la curva ROC global, tasa de falsos negativos de cada grupo, razón de
disparidad y **retorno en dólares** con los precios del laboratorio 14 (12,00 de contacto, 30 % de
retención, 104,42 de valor). Cierra con la pregunta difícil: si el remedio que más iguala es el que
menos dinero produce, ¿cuál se lleva al comité y cómo se defiende?

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: umbral por grupo → np.where(aud["tipo_cliente"]=="Mayorista", u_may, 0.50)
# Pista 2: el umbral que iguala la tasa de selección del mayorista es el cuantil correspondiente
#          de sus probabilidades: aud.loc[may,"prob"].quantile(1 - tasa_objetivo)
# Pista 3: usar umbrales distintos por grupo es legal en marketing y puede ser ILEGAL en crédito
#          o contratación. Escribe esa distinción en la respuesta, vale tanto como el código

### 🎯 Reto en clase (15 min)

En equipos y contra reloj: **consultoría cruzada**. Cada grupo entrega su modelo a otro grupo con una
sola hoja: qué predice, con qué variables y qué decisión desencadena. El grupo receptor tiene diez
minutos para escribir **tres objeciones**, una de cada tipo:

1. Una **variable sustituta** que sospechen que está dentro (y cómo probarlo con dos líneas de código).
2. Un **subgrupo** al que el modelo probablemente perjudica (y qué tabla habría que pedir).
3. Una **decisión** que el modelo no debería tomar aunque prediga bien (y por qué).

Después se devuelven las hojas y cada grupo tiene cinco minutos para contestar. **Las objeciones que no
puedan contestar van tal cual a la sección de limitaciones del proyecto.** Esa es la entrega real del
reto.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: para la objeción 1, se_puede_reconstruir() de la sección 4 vale casi sin tocarla
# Pista 2: para la objeción 2, metricas_subgrupo() aplicada a la variable que sospechen
# Pista 3: apunta las objeciones que NO pudiste contestar. Son la entrega, no el fracaso

## La trampa de hoy

⚠️ **Creer que quitar la variable sensible elimina el sesgo.** Es la trampa más extendida y la más
cómoda, porque permite declarar el problema resuelto sin medir nada. El código para desmontarla ya está
escrito: son los tres modelos de la sección 4, ahora puestos uno al lado del otro.

In [ ]:
def foto(aud_x, etiqueta):
    t = aud_x.groupby("tipo_cliente").apply(metricas_subgrupo, include_groups=False)
    return pd.Series({
        "¿está la columna tipo_cliente?": etiqueta,
        "área bajo la curva ROC": roc_auc_score(aud_x["real"], aud_x["prob"]),
        "TFN mayoristas": t.loc["Mayorista", "TFN · falsos negativos"],
        "TFN minoristas": t.loc["Minorista", "TFN · falsos negativos"],
        "razón de disparidad": (t["TFN · falsos negativos"].max()
                                / t["TFN · falsos negativos"].min()),
        "tasa de selección mayoristas": t.loc["Mayorista", "tasa de selección"],
        "tasa de selección minoristas": t.loc["Minorista", "tasa de selección"],
    })


trampa = pd.DataFrame([foto(aud, "SÍ, está dentro"),
                       foto(aud_sin_tipo, "NO, la quitamos"),
                       foto(aud_sin_nada, "NO, y tampoco la ciudad")],
                      index=["modelo completo", "sin tipo_cliente", "sin tipo_cliente ni ciudad"])
print(trampa.T.to_string(float_format=lambda v: f"{v:.4f}"))

print(f"\nEl número equivocado : «quitamos la variable sensible, el modelo ya no discrimina»")
print(f"El número correcto   : la tasa de falsos negativos de los mayoristas pasa de "
      f"{trampa.loc['modelo completo', 'TFN mayoristas']:.4f} a "
      f"{trampa.loc['sin tipo_cliente', 'TFN mayoristas']:.4f}")
print(f"                       y la de los minoristas de "
      f"{trampa.loc['modelo completo', 'TFN minoristas']:.4f} a "
      f"{trampa.loc['sin tipo_cliente', 'TFN minoristas']:.4f}")
print(f"                       la disparidad pasa de "
      f"{trampa.loc['modelo completo', 'razón de disparidad']:.4f} a "
      f"{trampa.loc['sin tipo_cliente', 'razón de disparidad']:.4f}")

📌 **Quitar la columna `tipo_cliente` deja la tasa de falsos negativos de los mayoristas exactamente
igual: 1,0000 antes y 1,0000 después.** El área bajo la curva se mueve de 0,7589 a 0,7587 —nada— y la
tasa de selección de los mayoristas sigue en 0,0096. El modelo no perdió la información: la tenía
repartida en el ticket medio, en el monto y en la frecuencia, tal como demostramos en la sección 4 con
un 99,82 % de exactitud usando solo el ticket.

Y cuando quitamos también la ciudad, la tasa de falsos negativos de los mayoristas baja de 1,0000 a
0,9474: **de los diecinueve mayoristas que se van, el modelo detecta uno**. Sigue siendo un fracaso completo
para ese grupo.

Qué sí funciona, en orden de esfuerzo:

1. **Medir por subgrupo.** No es un remedio, es el requisito. Un sesgo que no se mide no se puede
   discutir, y todos los modelos lo tienen.
2. **Cambiar el umbral por grupo** cuando la decisión lo permite. Es lo más barato y lo más efectivo, y
   es una decisión explícita que alguien firma.
3. **Cambiar la variable objetivo.** Si «180 días sin comprar» castiga al que compra por temporada, el
   problema no es el modelo: es la definición.
4. **Modelos separados por segmento** cuando los grupos se comportan de forma tan distinta que uno solo
   no puede servir a los dos. Cuesta más y hay que mantener dos.
5. **No desplegar.** Es una opción legítima y hay que tenerla sobre la mesa. Si el modelo perjudica
   sistemáticamente al grupo que más factura y no sabes arreglarlo, no sale del cuaderno.

## Entregable

Borrador completo del cuaderno reproducible del proyecto, con la sección de limitaciones y sesgos
escrita. Además, en `lab_15_apellido.ipynb`:

- El **modelo del caso reconstruido dentro del propio cuaderno**, sin depender de otro archivo, con su
  métrica global de referencia.
- La **tabla de desempeño por subgrupo** con el tamaño de cada grupo al lado, para al menos dos
  variables de agrupación.
- La **disparidad encontrada con su cifra**: tasa de selección, falsos positivos y falsos negativos, y
  la regla de las cuatro quintas partes aplicada. En Comercial Andina: 24,3 veces en tasa de selección
  entre mayorista y minorista, 3,40 veces en falsos positivos entre Cuenca y Loja.
- La prueba de **variable sustituta**: 0,9988 de exactitud reconstruyendo el tipo de cliente contra
  0,7783 de la línea base, y el contraejemplo de la ciudad, que no se puede reconstruir (0,3693 contra
  0,3746).
- La demostración de que **quitar la variable no arregla la disparidad**: 1,0000 de falsos negativos en
  mayoristas antes y después.
- La **importancia global** y una **explicación local** de un caso concreto, con la carta redactada en
  lenguaje de cliente y las tres preguntas contestadas: qué pasa cuando se equivoca, si se puede apelar
  y si la persona puede cambiar el resultado.
- La **lista de verificación de privacidad y gobernanza** llena para el proyecto del grupo, con
  evidencia en la columna de nota, no con un sí.
- Una fila nueva en la bitácora de prompts: pídele al asistente que haga de cliente perjudicado por tu
  modelo y que reclame. **Las objeciones que no puedas contestar van a la sección de limitaciones**, y
  se entregan tal cual.

## Para tu equipo

- La sección de limitaciones se escribe **antes** de la defensa, no durante. Un panel que encuentra una
  limitación que ustedes no escribieron asume que no la vieron; una limitación escrita por ustedes
  demuestra criterio y cierra la discusión.
- Busquen la variable sustituta de verdad, con el modelo auxiliar y su línea base. En un caso de
  recursos humanos suele ser el barrio o el colegio; en crédito, el código postal; en Comercial Andina
  es el ticket medio. **Si no encuentran ninguna, eso también se reporta con el número.**
- Decidan y escriban **qué error prefieren cometer**. No se pueden minimizar los dos, y el equipo que
  llega a la defensa sin esa frase escrita la va a improvisar delante del panel.